# Welcome to the Day 2 Lab!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get started --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## First - let's talk about the Chat Completions API

1. The simplest way to call an LLM
2. It's called Chat Completions because it's saying: "here is a conversation, please predict what should come next"
3. The Chat Completions API was invented by OpenAI, but it's so popular that everybody uses it!

### We will start by calling OpenAI again - but don't worry non-OpenAI people, your time is coming!


In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("gsk_"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


## Do you know what an Endpoint is?

If not, please review the Technical Foundations guide in the guides folder

And, here is an endpoint that might interest you...

In [2]:
import requests

headers = {"Authorization": f"Bearer {api_key}", 
           "Content-Type": "application/json"
          }

payload = {
    "model": "openai/gpt-oss-120b",
    "messages": [{"role":"system","content":"reply in markdown"},
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

{'model': 'openai/gpt-oss-120b',
 'messages': [{'role': 'system', 'content': 'reply in markdown'},
  {'role': 'user', 'content': 'Tell me a fun fact'}]}

In [3]:
response = requests.post(
    "https://api.groq.com/openai/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

{'id': 'chatcmpl-226ed667-23f5-46f9-ad9f-ff909aaa151c',
 'object': 'chat.completion',
 'created': 1789733271,
 'model': 'openai/gpt-oss-120b',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': '## 🎉 Fun Fact: Octopuses Have Three Hearts!\n\n- **Two branchial hearts** pump blood through the gills, where it picks up oxygen.  \n- **One systemic heart** circulates the oxygen‑rich blood to the rest of the body.  \n\nAnd that’s not all—octopus blood is **copper‑based (hemocyanin)**, which makes it appear blue and works better than iron‑based hemoglobin in cold, low‑oxygen water. 🐙✨\n\nSo, the next time you see an octopus, remember it’s literally running on three hearts!',
    'reasoning': 'The user asks for a fun fact. Provide a fun fact, maybe about space, biology, etc. In markdown. Probably one fun fact with some detail.'},
   'logprobs': None,
   'finish_reason': 'stop'}],
 'usage': {'queue_time': 0.403002612,
  'prompt_tokens': 82,
  'prompt_time': 0.004252315,

In [9]:
response.json()["choices"][0]["message"]["content"]

'## 🎉 Fun Fact\n\n**Octopuses have three hearts and blue blood!**  \n\n- Two hearts pump blood to the gills, while the third pumps it to the rest of the body.  \n- Their blood uses a copper‑based molecule called **hemocyanin**, which is more efficient than hemoglobin for transporting oxygen in cold, low‑oxygen water—hence the blue color.  \n\nSo the next time you see an octopus, remember it’s literally *heart‑felt* and *blue‑blooded* in more ways than one!'

# What is the openai package?

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!


In [6]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI(api_key=os.getenv("GROQ_API_KEY"),
                base_url="https://api.groq.com/openai/v1")

response = openai.chat.completions.create(model="openai/gpt-oss-120b", 
                                          messages=[{"role": "user", "content": "Tell me a 2 fun fact"}])

response.choices[0].message.content



'Here are two fun facts you might enjoy:\n\n1. **Octopuses have three hearts** – two pump blood to the gills, while the third circulates it to the rest of the body. Even more interesting, their blood is copper‑based, giving it a blue hue!\n\n2. **Bananas are berries, but strawberries aren’t** – In botanical terms, a berry is a fruit produced from a single ovary with seeds embedded throughout the flesh. By that definition, bananas (and even kiwis and grapes) qualify as berries, whereas strawberries develop from a flower with multiple ovaries, making them “aggregate fruits” rather than true berries.'

## And then this great thing happened:

OpenAI's Chat Completions API was so popular, that the other model providers created endpoints that are identical.

They are known as the "OpenAI Compatible Endpoints".

For example, google made one here: https://generativelanguage.googleapis.com/v1beta/openai/

And OpenAI decided to be kind: they said, hey, you can just use the same client library that we made for GPT. We'll allow you to specify a different endpoint URL and a different key, to use another provider.

So you can use:

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

And to be clear - even though OpenAI is in the code, we're only using this lightweight python client library to call the endpoint - there's no OpenAI model involved here.

If you're confused, please review Guide 9 in the Guides folder!

And now let's try it!

## THIS IS OPTIONAL - but if you wish to try out Google Gemini, please visit:

https://aistudio.google.com/

And set up your API key at

https://aistudio.google.com/api-keys

And then add your key to the `.env` file, being sure to Save the .env file after you change it:

`GOOGLE_API_KEY=AIz...`


In [16]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith(("AIz", "AQ.")):
    print("An API key was found, but it doesn't start with AIz or AQ.")
else:
    print("API key found and looks good so far!")



API key found and looks good so far!


In [17]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Here is a fun one for you: **Sea otters hold hands when they sleep.**\n\nThey do this to keep from drifting apart in the current while they nap. They often form "rafts" by holding onto each other, and sometimes they will even wrap themselves in long strands of giant kelp to act as an anchor so they stay in one place!'

## And Ollama also gives an OpenAI compatible endpoint

...and it's on your local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [4]:
requests.get("http://localhost:11434").content

b'Ollama is running'

### Download llama3.2 from meta

Change this to llama3.2:1b if your computer is smaller.

Don't use llama3.3 or llama4! They are too big for your computer..

In [ ]:
!ollama pull llama3.2

In [10]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [11]:
# Get a fun fact

response = ollama.chat.completions.create(model="llama3.1:8b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Here\'s one:\n\nThere is a species of jellyfish that is immortal!\n\nThe Turritopsis dohrnii, also known as the "immortal jellyfish," is a type of jellyfish that can transform its body into a younger state through a process called transdifferentiation. This means that it can essentially revert back to its polyp stage, which is the juvenile form of a jellyfish, and then grow back into an adult again. This process can be repeated indefinitely, making it theoretically immortal!\n\nIsn\'t that mind-blowing?'

In [ ]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

In [ ]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`

In [16]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
system_prompt = """
You are a helpful assistant that analyzes the contents of a website,
and provides a brief summary person who built the website, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""
user_prompt = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
"""
def message_for(website):

    return [{"role" : "system" , "content" : system_prompt},
            {"role" : "user" , "content" : user_prompt + website}]
def summarize (url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create (model = "llama3.1:8b",messages = message_for(website))
    return response.choices[0].message.content

print(summarize ("https://edwarddonner.com"))

print (f"\n\n\n{summarize ("https://portfoo07.netlify.app/")}")

**About the Website**
This website is a personal website for Edward Donner, a co-founder and CTO of AI startup Nebula.io.

**News/Announcements**

* Edward Donner released a series of blog posts with resources for AI coders, including:
	+ "AI Coder: Vibe Coder to Agentic Engineer"
	+ "AI Builder with n8n – Create Agents and Voice Agents"
	+ "AI Engineering MLOps Track – Deploy AI to Production"
	+ "Which order to take the AI courses?"
* All of these posts were published in 2025.

**Key Information**

* Edward Donner has experience as a founder and CEO of AI startup untapt (acquired in 2021) and a Managing Director at JPMorgan.
* He creates best-selling Udemy courses on AI-related topics, with over 900,000 enrollments.
* You can contact him through his website or find him on LinkedIn, Twitter, or Facebook.



# Portfolio Website of a React Developer
This website appears to be a personal portfolio built using the React framework, showcasing the developer's skills and projects. However, d